# 033 · Jersey-number detector: find the number before reading it (Google Colab)

**hoop-reid-tracker**, experiment 033.

### Why
The identity switches left in the pipeline happen at crossings, mostly between players the appearance models
cannot tell apart reliably inside the tracker (030, 031). Every serious work on player identity ends with the same
recipe: **read the jersey number, vote it over time, and use (team, number) to join the pieces**. Our first try
(023, Florence-2 zero-shot on the whole upper body) was right 96 % of the time when it answered, but it answered
for only **22 % of the pieces** on val: too few to matter (only 2 of the 37 wrong merges had a number on both sides).

What the working systems do and we did not (Roboflow "Basketball AI", among others): **detect the number region
first**, then read the tight crop. A text reader looking at a 30 x 20 px number is a much easier problem than one
looking for it in a whole torso.

### What this notebook does
1. Downloads a public basketball dataset from Roboflow Universe that has a **number** class (the jersey-number area)
   and **player** classes.
2. Turns it into the situation of our pipeline: one **player crop** per image (our detector already gives the
   players), with the number boxes that lie inside that player. Players without a visible number are kept as
   negatives (most crops in a game have no readable number).
3. Trains a small detector (YOLO26n, one class: `number`) on those crops.
4. Tests it on the dataset's test split and shows the tight crops the reader will get.

The reader itself is not trained here. On the laptop the detector's tight crops go to Florence-2 (zero-shot, as in
023), and coverage is measured on our 15 val clips. Only if coverage rises clearly (half of the pieces or more,
instead of 22 %) does it go into the stitching (`--jersey-rule`, already in the repo). If the reader then makes
too many mistakes, it is fine-tuned in a later step.

### Before running
* **Data**, either:
  * **(A, simplest)** download the dataset as a zip from its Roboflow Universe page (format "YOLOv8") and upload it
    to `MyDrive/hooptrack/number_det_src.zip`; or
  * **(B)** let the notebook download it: add your Roboflow API key as a Colab secret named `ROBOFLOW_API_KEY`
    (key icon in the left bar) and give the notebook access to it. The key is never printed.
* **Runtime → Change runtime type → T4 GPU**, then run all cells. Training: well under an hour on a T4
  (small model, small crops).

### Data, model and licences
| | |
|---|---|
| **Data in** | Roboflow Universe `roboflow-jvuqo/basketball-player-detection-3-ycjdo`, downloaded with **your own** API key (Colab secret `ROBOFLOW_API_KEY`); its licence is printed by section 3 from the export's README. |
| **Starting model** | YOLO26n, COCO weights (Ultralytics): **AGPL-3.0**. |
| **Save the result as** | `weights/number_detector_yolo26n.pt` (experiment 033, negative). |
| **Licence of the result** | AGPL-3.0 + the dataset's licence: do not redistribute it. |

Google Drive: `drive.mount` asks **you** to sign in and mounts **your own** Drive; `MyDrive/hooptrack/` is a folder in it.
Nothing personal is stored in this notebook. All sources and licences: [`docs/data_models_licences.md`](../docs/data_models_licences.md).


## 1 · GPU and libraries

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "ultralytics>=8.4" roboflow   # YOLO26 needs ultralytics 8.4 or newer

## 2 · Configuration
* `RF_WORKSPACE`, `RF_PROJECT`, `RF_VERSION`: the Roboflow Universe dataset (option B). The default is the dataset
  Roboflow uses in its basketball notebook; check the version number on its page.
* `NUMBER_CLASS` / `PLAYER_WORDS`: classes are found **by name**, not by index (datasets change between versions):
  the class whose name contains `number`, and the classes whose name contains one of `PLAYER_WORDS`.
* `PAD`: the player crop is enlarged by this fraction on each side, as a detector box is never exact.
* `IOS`: a number belongs to a player when at least this share of the number box is inside the player box
  (intersection over the SMALLER box; Roboflow uses 0.9).
* `MIN_HEIGHT`: smaller players are not used (their number is a few pixels: unreadable anyway).
* Training: `IMGSZ` 320 (crops are small), `EPOCHS` with early stop (`PATIENCE`), **`fliplr=0`**: a mirrored
  "2" or "7" is not a number, so the usual horizontal flip augmentation would teach wrong shapes.

In [ ]:
import os
from pathlib import Path

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/hooptrack")
    WORK = Path("/content")
else:                                   # local smoke test
    DRIVE_DIR = Path(os.environ.get("HOOPTRACK_DIR", "data"))
    WORK = Path(os.environ.get("HOOPTRACK_WORK", "/tmp/hooptrack"))

SRC_ZIP = DRIVE_DIR / "number_det_src.zip"          # option A: the Universe export, uploaded to Drive
RF_WORKSPACE = "roboflow-jvuqo"                     # option B: download with the API key
RF_PROJECT = "basketball-player-detection-3-ycjdo"
RF_VERSION = int(os.environ.get("HOOPTRACK_RF_VERSION", 1))
NUMBER_CLASS = "number"
PLAYER_WORDS = ("player",)                          # e.g. "player", "player-in-possession", ...
PAD, IOS, MIN_HEIGHT = 0.10, 0.9, 40

RUNS = (DRIVE_DIR / "runs").resolve()           # absolute: ultralytics puts relative paths under runs/detect/
NAME = "number_yolo26n_320"
MODEL = os.environ.get("HOOPTRACK_MODEL", "yolo26n.pt")
IMGSZ = int(os.environ.get("HOOPTRACK_IMGSZ", 320))
EPOCHS = int(os.environ.get("HOOPTRACK_EPOCHS", 80))
PATIENCE = 15
BATCH = int(os.environ.get("HOOPTRACK_BATCH", 32))
WORKERS = 2 if IN_COLAB else 0
RUNS.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
print(f"experiment 033 | model {MODEL} @ {IMGSZ} | runs {RUNS / NAME}")

## 3 · Get the dataset and check its licence and classes
Roboflow exports contain a `README.roboflow.txt` / `README.dataset.txt` with the licence: it is printed here so it
is on record (the repository will only keep the trained weights and the report, not the images).

In [ ]:
import shutil, zipfile, yaml

SRC = WORK / "number_det_src"
if not (SRC / "data.yaml").is_file():
    if SRC_ZIP.is_file():                                   # option A
        with zipfile.ZipFile(SRC_ZIP) as z:
            z.extractall(SRC)
    else:                                                   # option B
        from roboflow import Roboflow
        key = os.environ.get("ROBOFLOW_API_KEY")
        if key is None and IN_COLAB:
            from google.colab import userdata
            key = userdata.get("ROBOFLOW_API_KEY")
        assert key, "no dataset: upload number_det_src.zip to MyDrive/hooptrack/ or add the ROBOFLOW_API_KEY secret"
        project = Roboflow(api_key=key).workspace(RF_WORKSPACE).project(RF_PROJECT)
        project.version(RF_VERSION).download("yolov8", location=str(SRC), overwrite=True)
    # some exports put everything one folder down
    if not (SRC / "data.yaml").is_file():
        inner = next(SRC.rglob("data.yaml")).parent
        for p in inner.iterdir():
            shutil.move(str(p), SRC / p.name)

for readme in sorted(SRC.glob("README*.txt")):
    text = readme.read_text(errors="ignore")
    print(f"--- {readme.name}\n" + "\n".join(l for l in text.splitlines()
                                            if any(w in l.lower() for w in ("licen", "provided by", "url", "cc "))))

src_cfg = yaml.safe_load((SRC / "data.yaml").read_text())
NAMES = src_cfg["names"] if isinstance(src_cfg["names"], list) else [src_cfg["names"][k] for k in sorted(src_cfg["names"])]
NUM_IDS = [i for i, n in enumerate(NAMES) if NUMBER_CLASS in n.lower()]
PLAYER_IDS = [i for i, n in enumerate(NAMES) if any(w in n.lower() for w in PLAYER_WORDS) and "referee" not in n.lower()]
print("classes:", dict(enumerate(NAMES)))
print("number classes:", [NAMES[i] for i in NUM_IDS], "| player classes:", [NAMES[i] for i in PLAYER_IDS])
assert NUM_IDS, f"no class containing '{NUMBER_CLASS}': set NUMBER_CLASS to the right name"
assert PLAYER_IDS, f"no player class: set PLAYER_WORDS"

def split_dir(split):                       # Roboflow names: train / valid / test
    for name in (split, "val" if split == "valid" else split):
        if (SRC / name / "images").is_dir():
            return SRC / name
    return None

SPLITS = {s: split_dir(s) for s in ("train", "valid", "test")}
print({s: (len(list((d / "images").glob("*"))) if d else 0) for s, d in SPLITS.items()}, "images")

## 4 · From full frames to player crops (the pipeline's situation)
For every player box: crop it (enlarged by `PAD`), keep the number boxes that lie inside it (IoS >= `IOS`), and
write them in the crop's coordinates, as class 0. A player with no number inside is kept, with an empty label:
the detector must also learn to say "no number here" (backs turned away, numbers hidden by an arm).

In [ ]:
import cv2
import numpy as np

def yolo_boxes(label_file, w, h):
    # YOLO label file -> (classes, pixel boxes x1 y1 x2 y2)
    rows = [l.split() for l in label_file.read_text().splitlines() if l.strip()] if label_file.is_file() else []
    if not rows:
        return np.zeros(0, int), np.zeros((0, 4))
    r = np.array([[float(v) for v in row[:5]] for row in rows])
    c, cx, cy, bw, bh = r.T
    return c.astype(int), np.c_[(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h]

def ios(small, big):
    # share of each `small` box inside `big` (intersection over the area of the small box)
    x1 = np.maximum(small[:, 0], big[0]); y1 = np.maximum(small[:, 1], big[1])
    x2 = np.minimum(small[:, 2], big[2]); y2 = np.minimum(small[:, 3], big[3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    return inter / ((small[:, 2] - small[:, 0]) * (small[:, 3] - small[:, 1]) + 1e-9)

CROPS = WORK / "number_crops"
counts = {}
if not (CROPS / "data.yaml").is_file():
    for split, d in SPLITS.items():
        if d is None:
            continue
        out = "val" if split == "valid" else split
        (CROPS / "images" / out).mkdir(parents=True, exist_ok=True)
        (CROPS / "labels" / out).mkdir(parents=True, exist_ok=True)
        n_crops = n_with = n_numbers = 0
        for img_path in sorted((d / "images").glob("*")):
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            H, W = img.shape[:2]
            cls, boxes = yolo_boxes(d / "labels" / (img_path.stem + ".txt"), W, H)
            players = boxes[np.isin(cls, PLAYER_IDS)]
            numbers = boxes[np.isin(cls, NUM_IDS)]
            for k, (x1, y1, x2, y2) in enumerate(players):
                if y2 - y1 < MIN_HEIGHT:
                    continue
                px, py = PAD * (x2 - x1), PAD * (y2 - y1)
                cx1, cy1 = int(max(0, x1 - px)), int(max(0, y1 - py))
                cx2, cy2 = int(min(W, x2 + px)), int(min(H, y2 + py))
                crop = img[cy1:cy2, cx1:cx2]
                inside = numbers[ios(numbers, np.array([x1, y1, x2, y2])) >= IOS] if len(numbers) else numbers
                cw, ch = cx2 - cx1, cy2 - cy1
                lines = []
                for nx1, ny1, nx2, ny2 in inside:
                    nx1, ny1 = max(nx1, cx1) - cx1, max(ny1, cy1) - cy1
                    nx2, ny2 = min(nx2, cx2) - cx1, min(ny2, cy2) - cy1
                    lines.append(f"0 {(nx1 + nx2) / 2 / cw:.6f} {(ny1 + ny2) / 2 / ch:.6f} "
                                 f"{(nx2 - nx1) / cw:.6f} {(ny2 - ny1) / ch:.6f}")
                stem = f"{img_path.stem}_p{k}"
                cv2.imwrite(str(CROPS / "images" / out / f"{stem}.jpg"), crop)
                (CROPS / "labels" / out / f"{stem}.txt").write_text("\n".join(lines))
                n_crops += 1; n_with += bool(lines); n_numbers += len(lines)
        counts[out] = {"player_crops": n_crops, "crops_with_number": n_with, "numbers": n_numbers}
    (CROPS / "data.yaml").write_text(yaml.safe_dump(
        {"path": str(CROPS), "train": "images/train", "val": "images/val",
         **({"test": "images/test"} if (CROPS / "images" / "test").is_dir() else {}),
         "names": {0: "number"}}, sort_keys=False))
DATA = CROPS / "data.yaml"
print(counts or "crops already built")

### What the detector will learn from
Random player crops with their number boxes (green). Crops without a box are the negatives.

In [ ]:
import random
import matplotlib.pyplot as plt

random.seed(0)
imgs = sorted((CROPS / "images" / "train").glob("*.jpg"))
fig, axes = plt.subplots(2, 8, figsize=(16, 6))
for ax, p in zip(axes.ravel(), random.sample(imgs, min(16, len(imgs)))):
    img = cv2.imread(str(p)); h, w = img.shape[:2]
    _, b = yolo_boxes(CROPS / "labels" / "train" / (p.stem + ".txt"), w, h)
    for x1, y1, x2, y2 in b.astype(int):
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    ax.imshow(img[..., ::-1]); ax.axis("off")
plt.tight_layout(); plt.show()

## 5 · Training
Same structure as the player detector of 011: checkpoints on Drive, resumable after a disconnection. One class,
small model, small input; **no horizontal flip** (mirrored digits are not digits).

In [ ]:
import torch
from ultralytics import YOLO

last = RUNS / NAME / "weights" / "last.pt"
finished = last.is_file() and torch.load(last, map_location="cpu", weights_only=False).get("epoch", -1) == -1
if finished:
    print("Training already finished:", last.parent)
elif last.is_file():
    print("Resuming from", last)
    YOLO(str(last)).train(resume=True)
else:
    YOLO(MODEL).train(
        data=str(DATA), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, workers=WORKERS,
        project=str(RUNS), name=NAME, exist_ok=True, seed=0, plots=True, single_cls=True,
        fliplr=0.0, degrees=5.0, scale=0.4, translate=0.1, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
        mosaic=1.0, close_mosaic=10,
    )
BEST = RUNS / NAME / "weights" / "best.pt"
print("best weights:", BEST)

## 6 · Test: does it find the numbers?
mAP on the dataset's test split (or val if there is no test split), and the two numbers that matter here:

* **recall at confidence 0.5**: of the numbers that are there, how many are found (coverage);
* **false numbers per 100 crops without a number**: a box on a crop with no number sends noise to the reader.

In [ ]:
SPLIT = "test" if (CROPS / "images" / "test").is_dir() else "val"
model = YOLO(str(BEST))
m = model.val(data=str(DATA), split=SPLIT, imgsz=IMGSZ, conf=0.001, plots=False, verbose=False,
              project=str(WORK / "val_runs"), name="number", exist_ok=True)
metrics = {"split": SPLIT, "P": round(float(m.box.mp), 3), "R": round(float(m.box.mr), 3),
           "mAP50": round(float(m.box.map50), 3), "mAP50-95": round(float(m.box.map), 3)}

def iou(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

found = total = false_pos = empty = 0
test_imgs = sorted((CROPS / "images" / SPLIT).glob("*.jpg"))
for p in test_imgs:
    img = cv2.imread(str(p)); h, w = img.shape[:2]
    _, gt = yolo_boxes(CROPS / "labels" / SPLIT / (p.stem + ".txt"), w, h)
    pred = model.predict(str(p), imgsz=IMGSZ, conf=0.5, verbose=False)[0].boxes.xyxy.cpu().numpy()
    if len(gt):
        total += len(gt)
        found += int((iou(gt, pred).max(axis=1) >= 0.5).sum()) if len(pred) else 0
    else:
        empty += 1; false_pos += len(pred)
metrics.update({"recall_at_0.5": round(found / max(total, 1), 3),
                "false_numbers_per_100_empty_crops": round(100 * false_pos / max(empty, 1), 1),
                "test_crops": len(test_imgs), "test_numbers": total})
print(metrics)

### The tight crops the reader will get
For random test crops with a number: the box found (confidence >= 0.5), enlarged by 15 %, as it will be sent to
the reader on the laptop. If a human cannot read these, no reader will.

In [ ]:
with_num = [p for p in test_imgs if (CROPS / "labels" / SPLIT / (p.stem + ".txt")).read_text().strip()]
random.seed(1)
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
for ax, p in zip(axes.ravel(), random.sample(with_num, min(24, len(with_num)))):
    img = cv2.imread(str(p)); h, w = img.shape[:2]
    r = model.predict(str(p), imgsz=IMGSZ, conf=0.5, verbose=False)[0].boxes
    ax.axis("off")
    if not len(r):
        ax.set_title("not found", fontsize=8); ax.imshow(img[..., ::-1]); continue
    x1, y1, x2, y2 = r.xyxy.cpu().numpy()[r.conf.cpu().numpy().argmax()]
    px, py = 0.15 * (x2 - x1), 0.15 * (y2 - y1)
    tight = img[int(max(0, y1 - py)):int(min(h, y2 + py)), int(max(0, x1 - px)):int(min(w, x2 + px))]
    ax.imshow(tight[..., ::-1]); ax.set_title(f"{tight.shape[1]}x{tight.shape[0]} px", fontsize=8)
plt.tight_layout(); plt.savefig(RUNS / NAME / "tight_crops.jpg", dpi=80); plt.show()

## 7 · Save and take it home
1. Run the next cell: it copies the weights to `MyDrive/hooptrack/number_detector_yolo26n.pt` and writes
   `eval_033.json` (dataset, licence lines, classes used, counts, recipe, metrics).
2. Download both, put the weights in `weights/` and the report in `experiments/033_jersey_number_detector/`.
3. Next on the laptop: read the tight crops with Florence-2 on the 15 val clips and measure the coverage
   (the gate of 033).

In [ ]:
import json
shutil.copy(BEST, DRIVE_DIR / "number_detector_yolo26n.pt")
licence = {r.name: [l for l in r.read_text(errors="ignore").splitlines() if "licen" in l.lower()]
           for r in sorted(SRC.glob("README*.txt"))}
report = {"experiment": "033", "dataset": f"{RF_WORKSPACE}/{RF_PROJECT} v{RF_VERSION}" if not SRC_ZIP.is_file()
          else str(SRC_ZIP.name), "licence_lines": licence, "classes": NAMES,
          "number_classes": [NAMES[i] for i in NUM_IDS], "player_classes": [NAMES[i] for i in PLAYER_IDS],
          "crops": counts, "recipe": {"model": MODEL, "imgsz": IMGSZ, "epochs_max": EPOCHS, "patience": PATIENCE,
                                      "batch": BATCH, "pad": PAD, "ios": IOS, "min_height": MIN_HEIGHT,
                                      "fliplr": 0.0},
          "metrics": metrics}
(RUNS / NAME / "eval_033.json").write_text(json.dumps(report, indent=2))
shutil.copy(RUNS / NAME / "eval_033.json", DRIVE_DIR / "eval_033.json")
print(json.dumps(report, indent=2))